# Discover `.xlsx` Data Format

Use this notebook to inspect Excel workbooks before building a stable import
pipeline. The goal is to understand sheets, headers, field types, missing
values, and identifiers without silently changing the raw data.

## Setup

Keep the input path explicit. Change `xlsx_path` when exploring a different
workbook.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
xlsx_path = DATA_DIR / "interim" / "labeled_policy_text_manual_collection.xlsx"

xlsx_path

## Workbook Inventory

Start with workbook-level metadata before reading cells into pandas. This
helps identify hidden sheets, formula cells, dimensions, and sheet names.

In [ ]:
assert xlsx_path.exists(), f"Missing workbook: {xlsx_path}"

workbook = load_workbook(xlsx_path, read_only=True, data_only=False)

sheet_inventory = pd.DataFrame(
    [
        {
            "sheet_name": worksheet.title,
            "visible_state": worksheet.sheet_state,
            "max_row": worksheet.max_row,
            "max_column": worksheet.max_column,
        }
        for worksheet in workbook.worksheets
    ]
)

sheet_inventory

In [ ]:
selected_sheet = sheet_inventory.loc[0, "sheet_name"]
selected_sheet

## Raw Preview

Read a small window without assuming headers. This is the safest way to see
title rows, notes, merged headers, and whether the first visible row is really
the variable name row.

In [ ]:
raw_preview = pd.read_excel(
    xlsx_path,
    sheet_name=selected_sheet,
    header=None,
    nrows=5,
    dtype=object,
)

raw_preview

## Header Guess

Choose the row that appears to contain field names. For a plain rectangular
sheet this is usually row `0`, but policy collection files sometimes contain
titles or notes above the table.

In [ ]:
header_row = 0

df = pd.read_excel(
    xlsx_path,
    sheet_name=selected_sheet,
    header=header_row,
    dtype=object,
)

df = df.dropna(how="all").reset_index(drop=True)
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.to_frame(index=False, name="raw_column_name")

In [ ]:
unnamed_columns = [
    column for column in df.columns if str(column).startswith("Unnamed:")
]

df[unnamed_columns].head() if unnamed_columns else "No unnamed columns found."

## Column Profile

Profile every column before deciding types. Excel often mixes dates, numeric
identifiers, and free text in the same column. Treat identifiers as strings
until you are certain leading zeros and suffixes do not matter.

In [ ]:
profile = pd.DataFrame(
    {
        "column": df.columns,
        "pandas_dtype": [df[column].dtype for column in df.columns],
        "non_null": [df[column].notna().sum() for column in df.columns],
        "missing": [df[column].isna().sum() for column in df.columns],
        "unique": [df[column].nunique(dropna=True) for column in df.columns],
        "example_values": [
            df[column].dropna().astype(str).head(5).tolist()
            for column in df.columns
        ],
    }
)

profile

## Duplicate and Blank Rows

Check whether the workbook includes repeated records, separators, or empty
rows. For policy text mining, duplicate policy titles or URLs may be valid
only if they refer to different issuing agencies or versions.

In [ ]:
blank_rows = df.isna().all(axis=1)
print(f"Blank rows after header parsing: {blank_rows.sum()}")

duplicate_rows = df.duplicated(keep=False)
print(f"Fully duplicated rows: {duplicate_rows.sum()}")

df.loc[duplicate_rows].head(10)

## Candidate Identifier Columns

Look for fields that may uniquely identify policies, firms, source pages, or
manual labels. A good import pipeline should preserve these as strings.

In [ ]:
candidate_id_terms = ("id", "code", "no", "number", "url", "link", "title", "name")
candidate_id_terms += ("编号", "代码", "编码", "网址", "链接", "标题", "名称", "企业")

candidate_id_columns = [
    column
    for column in df.columns
    if any(term in str(column).lower() for term in candidate_id_terms)
]

id_profile = pd.DataFrame(
    [
        {
            "column": column,
            "non_null": df[column].notna().sum(),
            "unique": df[column].nunique(dropna=True),
            "duplicated_non_null": df[column].dropna().duplicated().sum(),
            "examples": df[column].dropna().astype(str).head(5).tolist(),
        }
        for column in candidate_id_columns
    ]
)

id_profile

## Text Columns

Policy text mining depends on title, body, label, and instrument fields. This
cell finds columns that look text-heavy and reports length summaries.

In [ ]:
text_profile = []

for column in df.columns:
    values = df[column].dropna().astype(str).str.strip()
    if values.empty:
        continue

    lengths = values.str.len()
    text_profile.append(
        {
            "column": column,
            "mean_length": lengths.mean(),
            "max_length": lengths.max(),
            "share_length_over_100": (lengths > 100).mean(),
            "examples": values.head(3).tolist(),
        }
    )

pd.DataFrame(text_profile).sort_values("max_length", ascending=False)

## Date Candidates

Dates are central for staggered DID timing. Inspect parse success before
converting columns permanently.

In [ ]:
date_terms = ("date", "time", "year", "published", "issued", "release")
date_terms += ("日期", "时间", "年份", "年度", "发布日期", "发布时间", "印发日期")

date_candidates = [
    column
    for column in df.columns
    if any(term in str(column).lower() for term in date_terms)
]

date_parse_report = []

for column in date_candidates:
    raw_values = df[column].astype(str)
    if any(term in str(column).lower() for term in ("year", "年份", "年度")):
        extracted_year = raw_values.str.extract(r"(\d{4})", expand=False)
        parsed = pd.to_datetime(extracted_year, format="%Y", errors="coerce")
    else:
        parsed = pd.to_datetime(df[column], errors="coerce")
    date_parse_report.append(
        {
            "column": column,
            "non_null": df[column].notna().sum(),
            "parsed": parsed.notna().sum(),
            "min": parsed.min(),
            "max": parsed.max(),
            "unparsed_examples": df.loc[
                df[column].notna() & parsed.isna(), column
            ].astype(str).head(5).tolist(),
        }
    )

pd.DataFrame(date_parse_report)

## Value Counts for Low-Cardinality Fields

Manual labels, province names, policy instrument types, and source categories
should often have controlled vocabularies. Review low-cardinality columns for
spelling variants before modeling.

In [ ]:
low_cardinality_columns = [
    column
    for column in df.columns
    if 1 < df[column].nunique(dropna=True) <= 20
]

for column in low_cardinality_columns:
    print(f"\n## {column}")
    display(df[column].value_counts(dropna=False).head(30).to_frame("count"))

## Draft Cleaning Map

Do not overwrite raw columns during discovery. Draft a rename map and type
plan, then move stable decisions into a script or package function later.

In [ ]:
rename_map = {
    # "Raw Column Name": "snake_case_name",
}

type_notes = {
    # "snake_case_name": "string/date/category/int; explain why",
}

rename_map, type_notes

## Cleaned Preview

Use this as a sandbox for import choices. Keep this cell lightweight; once the
cleaning rules settle, move them into `src/` or `scripts/` and add tests.

In [ ]:
cleaned = df.rename(columns=rename_map).copy()

for column in cleaned.columns:
    if cleaned[column].dtype == object:
        cleaned[column] = cleaned[column].where(
            cleaned[column].isna(),
            cleaned[column].astype(str).str.strip(),
        )

cleaned.head()

In [ ]:
cleaned.info()

## Export Checklist

Before converting Excel to a model-ready file:

- Confirm which row is the true header.
- Preserve raw source files in `data/raw/` or `data/interim/`.
- Record source metadata in `data/source-manifest.csv`.
- Keep policy identifiers, firm identifiers, stock codes, and URLs as strings.
- Convert dates only after checking parse failures.
- Document final field meanings in `docs/data-dictionary.md`.
- Write derived rectangular data to `data/processed/` or `data/interim/`,
  not over the original workbook.